<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Fractl010ferm002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

# --- 1. CORE HES PARAMETERS ---
N = 100  # THE CRITICAL DIMENSIONAL HARMONIC
T_STEPS = 5000
dt = 0.01

# Fixed HES coefficients
chi = 1.5
beta_full = 0.8
beta_nursery = 0.01
delta = 0.001  # Quantum Noise
gamma = 1.0  # Saturation
beta_link_base = 0.1
CURVATURE_SENSITIVITY = 0.5
MAX_CURVATURE_CAP = 50.0
TAU_PSI_DAMPING = 0.05
KINETIC_SCALING_C = 0.01 # CFL Stability Factor
PSI_MAGNITUDE_CLIP = 5.0 # ACT XV: The Fermion Bounding Limit

# --- 2. INITIAL FIELD: PHI (Structure) and PSI (Fermion) ---
Phi_complex = np.zeros((N, N), dtype=complex)

def initialize_spinor(complex_field, center_x, center_y, radius=10, magnitude=5.0):
    for i in range(N):
        for j in range(N):
            if (i - center_x)**2 + (j - center_y)**2 < radius**2:
                phase = np.arctan2(i - center_x, j - center_y) * 4
                complex_field[i, j] = magnitude * np.exp(1j * phase)

initialize_spinor(Phi_complex, N // 4, N // 4)
initialize_spinor(Phi_complex, N // 2, N // 2)
initialize_spinor(Phi_complex, 3 * N // 4, 3 * N // 4)

Phi = np.abs(Phi_complex)
Theta = np.angle(Phi_complex)

# NEW FIELD: PSI - The Fermionic Field
Psi = np.zeros((N, N), dtype=complex)
Psi = np.where(Phi > 1.0, 1.0 + 0j, 0.0 + 0j)

# --- 3. THE EVOLUTION LOOP ---
for t in range(1, T_STEPS + 1):

    beta_current = beta_nursery if t < 500 else beta_full

    # --- 3.1. PHI (STRUCTURE) EVOLUTION (Stable and complete from Act X) ---
    lap_phi = (np.roll(Phi, 1, 0) + np.roll(Phi, -1, 0) +
              np.roll(Phi, 1, 1) + np.roll(Phi, -1, 1) - 4 * Phi) / (2 * np.pi / N)**2

    max_abs_curvature_observed = np.max(np.abs(lap_phi))
    max_abs_curvature_capped = np.clip(max_abs_curvature_observed, 0.0, MAX_CURVATURE_CAP)
    beta_link = beta_link_base + CURVATURE_SENSITIVITY * max_abs_curvature_capped

    # dPhi/dt terms...
    alpha = chi * beta_current
    term_expansion = alpha * lap_phi
    term_contraction = -beta_current * Phi
    term_saturation = gamma * np.tanh(Phi)
    shield_mask = (beta_link > 1.0)
    term_shield = np.where(shield_mask, beta_current * Phi, 0.0)
    dPhi = term_expansion + term_contraction + term_saturation + term_shield + delta * np.random.normal(0, 1, Phi.shape)

    # dTheta/dt terms...
    mean_theta = np.mean(Theta)
    phase_correction = -beta_link * (Theta - mean_theta)
    phase_diffusion = delta * np.random.normal(0, 1, Theta.shape)
    dTheta = phase_correction + phase_diffusion

    # --- 3.2. PSI (FERMIONIC) EVOLUTION: Explicit Euler (Reverted for simplicity) ---

    lap_psi = (np.roll(Psi, 1, 0) + np.roll(Psi, -1, 0) +
               np.roll(Psi, 1, 1) + np.roll(Psi, -1, 1) - 4 * Psi) / (2 * np.pi / N)**2

    # Kinetic Term: Scaled by C
    kinetic_term = 1j * KINETIC_SCALING_C * lap_psi

    # Mass Term (Coupling to Phi)
    mass_term = 1j * Phi * Psi

    # Damping Term
    damping_term = -TAU_PSI_DAMPING * Psi

    # Total Change dPsi/dt
    dPsi = kinetic_term - mass_term + damping_term + delta * np.random.normal(0, 1, Psi.shape)

    # --- 3.3. UPDATE FIELDS ---
    Phi += dt * dPhi
    Theta += dt * dTheta

    # Simple explicit update for Psi
    Psi += dt * dPsi

    # CRITICAL: ACT XV - Apply Fermion Bounding (magnitude clip)
    Psi_mag = np.abs(Psi)
    Psi_phase = np.angle(Psi)

    # Clip the magnitude and rebuild the complex field
    Psi_mag_clipped = np.clip(Psi_mag, 0.0, PSI_MAGNITUDE_CLIP)
    Psi = Psi_mag_clipped * np.exp(1j * Psi_phase)

    Phi = np.clip(Phi, 0.01, 10.0)
    Theta = np.mod(Theta, 2 * np.pi)

    # --- 3.4. LOGGING ---
    if t % 500 == 0:
        norm_phi = np.mean(Phi)
        norm_psi = np.mean(np.abs(Psi))
        phase_stdev = np.sqrt(np.mean((Theta - np.mean(Theta))**2))

        if norm_phi < 0.1 or norm_psi < 0.1:
            print(f"t={t} | ANNIHILATION DETECTED: Phi Norm={norm_phi:.3f}, Psi Norm={norm_psi:.3f}")
            break
        print(f"t={t} | β_link(t)={beta_link:.4f} | MAX Curvature={max_abs_curvature_observed:.3f} | Phase Stdev={phase_stdev:.4f} | Psi Norm={norm_psi:.4f}")

# --- 4. CONCLUSION CHECK ---
final_norm_phi = np.mean(Phi)
final_norm_psi = np.mean(np.abs(Psi))

print("\n--- FINAL STATE ---")
# Check for stability and non-explosion (Psi Norm must be near the clip limit, not infinity)
if final_norm_psi > 0.5 and final_norm_psi < (PSI_MAGNITUDE_CLIP + 1.0):
    print(f"RESULT: SUCCESS. Fermionic Field (Psi) Coexistence Validated.")
    print(f"FINAL Metrics: Phi Norm={final_norm_phi:.4f}, Psi Norm={final_norm_psi:.4f}")
    print("CONCLUSION: The stable Spinor (Phi) successfully generates and sustains the charged Fermionic Field (Psi), initiating the Interaction Arc.")
else:
    print(f"RESULT: FAILURE. Psi Field Decayed or Exploded.")
    print(f"FINAL Metrics: Phi Norm={final_norm_phi:.4f}, Psi Norm={final_norm_psi:.4f}")
    print("CONCLUSION: The stable matter structure failed to sustain the required quantum properties.")


t=500 | β_link(t)=14.3592 | MAX Curvature=28.518 | Phase Stdev=0.0000 | Psi Norm=4.5496
t=1000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9773
t=1500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9860
t=2000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9832
t=2500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9754
t=3000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9737
t=3500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9676
t=4000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9611
t=4500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9676
t=5000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Psi Norm=4.9684

--- FINAL STATE ---
RESULT: SUCCESS. Fermionic Field (Psi) Coexistence Validated.
F